In [1]:
import torch
from torch import nn, optim
from unet import Unet, dice_loss
from dataset import dataloader
from aug import load_augmentation
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import os
import json


In [2]:
cwd = Path(os.getcwd()).parent.parent
cwd

WindowsPath('e:/pythonReps')

In [8]:
import albumentations as A
from albumentations.pytorch import ToTensorV2


TARGET_SIZE = (256, 256)  # gewünschte Zielgröße

aug = A.Compose([
    A.Resize(height=TARGET_SIZE[0], width=TARGET_SIZE[1]),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    ToTensorV2()
])

In [9]:
# parser = argparse.ArgumentParser()
# parser.add_argument("--augmentation", type=str, default="baseline")
# parser.add_argument("--run_name",     type=str, required=True)
# parser.add_argument("--epochs",       type=int, default=100)
# parser.add_argument("--bs",           type=int, default=8)
# parser.add_argument("--lr",           type=float, default=1e-3)
# args = parser.parse_args()
# args = {"augmentation": "baseline"}

img_path      = cwd / 'Projektpraktikum_Master/augmentation_testing/dat/train/img'
mask_path     = cwd / 'Projektpraktikum_Master/augmentation_testing/dat/train/mask'
img_val_path  = cwd / 'Projektpraktikum_Master/augmentation_testing/dat/val/img'
mask_val_path = cwd / 'Projektpraktikum_Master/augmentation_testing/dat/val/mask'

out_dir = Path("res") / "test"
out_dir.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# aug = load_augmentation("hist_equilization", ref_dir=img_val_path)

train_dataloader = dataloader(img_path, mask_path, transform=aug, bs=8, shuffle=True, max_samples=50)

In [10]:
import matplotlib.pyplot as plt
import numpy as np

def show_samples(dataloader, n=4):
    images, masks = next(iter(dataloader))  # einen Batch holen

    fig, axes = plt.subplots(2, n, figsize=(n * 4, 8))
    fig.suptitle("Stichproben aus dem DataLoader", fontsize=14)

    for i in range(n):
        # Bild: (C, H, W) → (H, W, C) für matplotlib
        img = images[i].permute(1, 2, 0).numpy()
        
        # Normalisierung rückgängig machen (falls nötig)
        img = np.clip(img, 0, 1)

        # Maske: (1, H, W) oder (H, W)
        mask = masks[i].squeeze().numpy()

        axes[0, i].imshow(img)
        axes[0, i].set_title(f"Bild {i+1}\n{img.shape}")
        axes[0, i].axis("off")

        axes[1, i].imshow(mask, cmap="gray")
        axes[1, i].set_title(f"Maske {i+1}")
        axes[1, i].axis("off")

    plt.tight_layout()
    plt.show()

show_samples(train_dataloader, n=4)

KeyError: 'You have to pass data to augmentations as named arguments, for example: aug(image=image)'